# Image Captioning — Model Prototype
Works on **RunPod**, **Kaggle**, and **local Windows/Linux**. Run cells top-to-bottom.

## 0. Detect platform

In [ ]:
import platform, sys, os

IS_WINDOWS = platform.system() == "Windows"
IS_RUNPOD  = os.path.exists("/workspace")
IS_KAGGLE  = os.path.exists("/kaggle")

if IS_RUNPOD:
    PROJECT = "/workspace/Pattern_Recog_Project"
elif IS_KAGGLE:
    PROJECT = "/kaggle/working/Pattern_Recog_Project"
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(f"Platform : {platform.system()}")
print(f"RunPod   : {IS_RUNPOD}")
print(f"Kaggle   : {IS_KAGGLE}")
print(f"Project  : {PROJECT}")

## 1. Pull latest code & install deps

In [ ]:
import subprocess

if IS_KAGGLE:
    os.makedirs("/kaggle/working", exist_ok=True)
    if not os.path.exists(PROJECT):
        subprocess.run(["git", "clone", "https://github.com/RATSAPORN/Pattern_Recog_Project.git", PROJECT], check=True)
    else:
        subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
elif IS_RUNPOD:
    subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
else:
    print("Local machine — skipping git pull.")

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

In [ ]:
%pip install timm pycocoevalcap pycocotools matplotlib -q

## 2. Config — edit before running

In [ ]:
CHECKPOINT  = "models/best.pt"      # path to saved checkpoint
ENCODER     = "vmamba_tiny"          # must match training: vit_base | vit_small | vmamba_tiny | vmamba_small | ...
DECODER     = "transformer"          # transformer | mamba | mamba3
DECODER_DIM = 512
NUM_LAYERS  = 3
MAX_LEN     = 50

IMAGE_DIR   = "data/flickr8k/images/Flicker8k_Dataset"
ANN_JSON    = "data/flickr8k_annotations.json"
TEST_SPLIT  = "data/flickr8k/text/Flickr_8k.testImages.txt"   # 1,000 held-out test images

N_IMAGES    = 12   # number of random test images to show
COLS        = 3    # columns in the grid
SEED        = 42   # change for different random samples

## 3. Load model

In [ ]:
import torch
from src.models.predict import load_model, load_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

model, vocab = load_model(
    checkpoint_path=CHECKPOINT,
    encoder_name=ENCODER,
    decoder_name=DECODER,
    vocab_path=None,        # reads vocab_path from checkpoint automatically
    decoder_dim=DECODER_DIM,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    device=DEVICE,
)

## 4. Load ground truth captions

In [ ]:
import json
from pathlib import Path

# Ground truth captions for all images
with open(ANN_JSON) as f:
    gt_captions = json.load(f)   # {filename: [caption1, caption2, ...]}

# Load test split — only these 1,000 images were never seen during training
with open(TEST_SPLIT) as f:
    test_images = set(line.strip() for line in f if line.strip())

print(f"Total images with captions : {len(gt_captions)}")
print(f"Test split size            : {len(test_images)} images (never seen during training)")

## 5. Random prediction grid

In [ ]:
import random
import textwrap
import matplotlib.pyplot as plt
from PIL import Image

# Only pick from test split images (never seen during training)
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_paths = [
    p for p in Path(IMAGE_DIR).iterdir()
    if p.suffix.lower() in exts and p.name in test_images
]

random.seed(SEED)
chosen = random.sample(test_paths, min(N_IMAGES, len(test_paths)))
print(f"Showing {len(chosen)} random images from test split")

rows = (len(chosen) + COLS - 1) // COLS
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 5.5))
axes = axes.flatten() if rows * COLS > 1 else [axes]

for i, path in enumerate(chosen):
    pil_img = Image.open(path).convert("RGB")
    tensor  = load_image(str(path)).to(DEVICE)

    with torch.no_grad():
        pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

    fname = path.name
    gt = gt_captions.get(fname, ["(no ground truth)"])[0]

    axes[i].imshow(pil_img)
    axes[i].axis("off")

    pred_wrapped = "\n".join(textwrap.wrap(f"Pred: {pred}", width=40))
    gt_wrapped   = "\n".join(textwrap.wrap(f"GT:   {gt}",   width=40))

    axes[i].set_title(f"{pred_wrapped}\n{gt_wrapped}", fontsize=8, loc="left",
                      color="white",
                      bbox=dict(facecolor="black", alpha=0.6, pad=3))

for j in range(len(chosen), len(axes)):
    axes[j].axis("off")

plt.suptitle(f"{ENCODER} + {DECODER}  |  test split  |  seed={SEED}", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("predictions.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved → predictions.png")

## 7. Single image — detailed view

## 6. Evaluate on full test split

In [ ]:
from tqdm import tqdm
from src.models.train import compute_metrics

hypotheses = {}   # {image_id: [predicted_caption]}
references  = {}  # {image_id: [gt_caption1, gt_caption2, ...]}

model.eval()
for idx, path in enumerate(tqdm(test_paths, desc="evaluating test set")):
    fname  = path.name
    tensor = load_image(str(path)).to(DEVICE)

    with torch.no_grad():
        pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

    gts = gt_captions.get(fname, [""])

    hypotheses[str(idx)] = [pred]
    references[str(idx)]  = gts          # all 5 ground truth captions

metrics = compute_metrics(hypotheses, references)

print("\n── Test Set Metrics ─────────────────────")
for name, score in metrics.items():
    print(f"  {name:<10} {score:.4f}")
print("─────────────────────────────────────────")

In [ ]:
# Pick any image path here
SINGLE_IMAGE = str(chosen[0])

pil_img = Image.open(SINGLE_IMAGE).convert("RGB")
tensor  = load_image(SINGLE_IMAGE).to(DEVICE)

with torch.no_grad():
    pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

fname = Path(SINGLE_IMAGE).name
gts   = gt_captions.get(fname, ["(no ground truth)"])

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(pil_img)
ax.axis("off")

caption_text = f"Predicted:\n  {pred}\n\nGround truth (all 5):"
for k, gt in enumerate(gts[:5], 1):
    caption_text += f"\n  {k}. {gt}"

fig.text(0.5, -0.02, caption_text, ha="center", va="top",
         fontsize=10, wrap=True,
         bbox=dict(facecolor="lightyellow", alpha=0.8, pad=6))

plt.tight_layout()
plt.savefig("single_prediction.png", bbox_inches="tight", dpi=120)
plt.show()
print(f"Image   : {fname}")
print(f"Predicted: {pred}")
for k, gt in enumerate(gts[:5], 1):
    print(f"GT {k}     : {gt}")